# Ноутбук 6. Оценка бизнес-ценности политик коммуникации

## О чём этот ноутбук

В предыдущих ноутбуках мы построили несколько подходов к выбору клиентов для коммуникации:
классические модели (logistic regression, CatBoost) и uplift meta-learners (S, T, X, DR-Learner).

Все они давали разные значения AUUC и Spearman rho. Но что это означает **на практике**?

> *Если мы перейдём с risk-based стратегии на uplift-модель, сколько дополнительных дефолтов
> мы предотвратим при том же CRM-бюджете?*

На этот вопрос отвечает данный ноутбук. Мы переводим абстрактные метрики в конкретные
бизнес-результаты.

### Структура анализа

1. **Экономическая постановка** - стоимость каждого вида коммуникации и ценность предотвращённого дефолта.
2. **Политики коммуникации** - формальное определение семи сравниваемых стратегий.
3. **Кривые бюджет-эффект** - при каждом уровне затрат, сколько дефолтов предотвращает каждая стратегия.
4. **Multi-treatment policy** - не просто 'контактировать / нет', а какой канал выбрать для каждого клиента.
5. **Сегментный анализ** - Persuadables, Sleeping Dogs, Sure Things, Lost Causes.
6. **Bootstrap CI** - статистическая значимость различий между стратегиями.
7. **Сводные выводы и рекомендации** - когда стоит применять uplift-подход.


## 0. Импорты и настройки


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

RANDOM_SEED = 91
np.random.seed(RANDOM_SEED)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
pd.options.display.float_format = '{:.4f}'.format


## 1. Загрузка предсказанных скоров

Загружаем скоры, сохранённые в конце `uplift_models.ipynb`.

Каждая строка - один клиент из test или OOT выборки. Колонки:
- `y` - наблюдаемый outcome (1 = дефолт, 0 = нет)
- `treatment_bin` - получил ли клиент коммуникацию (1 = да)
- `BASE_PD` - базовая вероятность дефолта (для risk-based стратегии)
- `TRUE_UPLIFT` - истинный causal effect (оракульная переменная)
- `score_s/t/x/dr` - скоры ранжирования по каждому uplift-методу
- `uplift_s/t_*` - per-channel uplift (для multi-treatment policy)

> **Если файл не найден:** сначала запустите `uplift_models.ipynb` полностью.


In [ ]:
import os

SCORES_PATH = 'data/processed/uplift_scores.csv'

if not os.path.exists(SCORES_PATH):
    raise FileNotFoundError(
        'Файл uplift_scores.csv не найден. Запустите uplift_models.ipynb до конца.'
    )

df_scores = pd.read_csv(SCORES_PATH)
print(f'Загружено: {len(df_scores):,} строк x {df_scores.shape[1]} колонок')
print(f'test: {(df_scores.split == "test").sum():,}  |  OOT: {(df_scores.split == "oot").sum():,}')
print(f'Доля дефолтов: {df_scores.y.mean():.3%}')
print(f'Колонки: {list(df_scores.columns)}')


In [ ]:
# Разделяем на test и OOT для последующего анализа
df_test = df_scores[df_scores.split == 'test'].reset_index(drop=True)
df_oot  = df_scores[df_scores.split == 'oot'].reset_index(drop=True)

print(f'test: {len(df_test):,} клиентов | OOT: {len(df_oot):,} клиентов')


## 2. Экономическая постановка задачи

Для сравнения стратегий нам нужно задать **стоимости** и **выгоды** в единых единицах.

### Параметры модели

| Параметр | Значение | Обоснование |
|---|---|---|
| SMS-сообщение | 5 руб. | Рассылка по рыночным ценам |
| Автозвонок (robot) | 30 руб. | Стоимость платформы + телефония |
| Звонок оператора | 200 руб. | Среднее время 5 мин, ставка 2400 руб/ч |
| Средняя сумма кредита | 600,000 руб. | Статистика по портфелю |
| LGD (Loss Given Default) | 40% | Типичный показатель для потребительских кредитов |
| Ценность предотвращённого дефолта | 240,000 руб. | LGD * средняя сумма кредита |

### Принцип расчёта

Для каждого клиента, которому мы назначили коммуникацию типа $t$:

$$\text{ROI}_t(x) = \underbrace{(-\hat{\tau}_t(x)) \cdot V_{default}}_{\text{ожидаемая выгода}} - \underbrace{C_t}_{\text{стоимость коммуникации}}$$

где $-\hat{\tau}_t(x)$ - снижение вероятности дефолта (положительное = хорошо),
$V_{default}$ - ценность предотвращённого дефолта, $C_t$ - стоимость канала.

**Важная оговорка:** в нашей оценке мы используем *истинный* `TRUE_UPLIFT`, а не предсказанный.
Это позволяет честно оценить, насколько хорошо каждая *стратегия отбора* находит клиентов
с реальным эффектом - независимо от калибровки предсказаний.


In [ ]:
# Параметры экономической модели
COMM_COSTS = {
    'control':      0,
    'sms':          5,
    'robot_call':   30,
    'operator_call': 200,
}

# Средняя стоимость коммуникации (при смешанном составе treatment в данных)
COST_ANY_CONTACT = 80   # средневзвешенная по составу treatment-групп

LOSS_GIVEN_DEFAULT = 240_000   # руб.

print('Экономические параметры:')
for ch, cost in COMM_COSTS.items():
    if ch != 'control':
        print(f'  {ch:<20} стоимость={cost:>5} руб.')
print(f'  LGD (ценность предотвращённого дефолта): {LOSS_GIVEN_DEFAULT:,} руб.')

# Вычисляем предотвращённый дефолт по истинному uplift
# TRUE_UPLIFT < 0 означает снижение PD, поэтому берём -TRUE_UPLIFT
df_test['prevented_default_prob'] = -df_test['TRUE_UPLIFT']   # снижение вероятности дефолта
df_oot['prevented_default_prob']  = -df_oot['TRUE_UPLIFT']

print(f'\nРаспределение TRUE_UPLIFT (test):')
print(df_test['TRUE_UPLIFT'].describe().round(5))


## 3. Семь стратегий коммуникации

Для сравнения определяем семь стратегий. Каждая задаёт **ранжирование** клиентов:
кого контактировать в первую очередь при данном бюджете.

| # | Стратегия | Скор ранжирования | Тип |
|---|---|---|---|
| 1 | Никого не контактировать | 0 | Нижняя граница |
| 2 | Случайный выбор | rand() | Baseline |
| 3 | Risk-based | BASE_PD | Текущая практика |
| 4 | S-Learner | predicted uplift | Uplift |
| 5 | T-Learner | predicted uplift | Uplift |
| 6 | X-Learner | predicted uplift | Uplift |
| 7 | DR-Learner | predicted uplift | Uplift |
| 8 | Oracle | TRUE_UPLIFT | Верхняя граница |

**Механика оценки:** при бюджете B мы берём top-K клиентов по скору,
"назначаем" им коммуникацию и суммируем реальный `prevented_default_prob * LGD - cost`.


In [ ]:
# Определяем скоры для каждой стратегии
# Знак: выше скор = приоритетнее для контакта
rng = np.random.RandomState(RANDOM_SEED)

strategies = {
    'Никого не контактировать': np.zeros(len(df_test)),
    'Случайный выбор':           rng.rand(len(df_test)),
    'Risk-based (BASE_PD)':      df_test['BASE_PD'].values,
    'S-Learner':                 df_test['score_s'].values,
    'T-Learner':                 df_test['score_t'].values,
    'X-Learner':                 df_test['score_x'].values,
    'DR-Learner':                df_test['score_dr'].values,
    'Oracle':                   -df_test['TRUE_UPLIFT'].values,
}

# Цвета для визуализации
strategy_colors = {
    'Никого не контактировать': 'lightgray',
    'Случайный выбор':          'silver',
    'Risk-based (BASE_PD)':     'dimgray',
    'S-Learner':                'steelblue',
    'T-Learner':                'darkorange',
    'X-Learner':                'green',
    'DR-Learner':               'crimson',
    'Oracle':                   'black',
}
strategy_styles = {
    'Никого не контактировать': ':',
    'Случайный выбор':          ':',
    'Risk-based (BASE_PD)':     '--',
    'S-Learner':  '-',
    'T-Learner':  '-',
    'X-Learner':  '-',
    'DR-Learner': '-',
    'Oracle':     '--',
}

print('Стратегии определены:', list(strategies.keys()))


## 4. Кривые "бюджет - предотвращённые дефолты"

**Основная визуализация ноутбука.** По оси X - доля клиентов, которых мы контактируем
(прокси бюджета: больше клиентов = больше затрат). По оси Y - суммарное снижение
вероятности дефолтов среди выбранных клиентов.

### Как читать этот график

- **Oracle** (чёрная пунктирная) - теоретический максимум. Никакая реальная модель не может
  подняться выше.
- **Risk-based** (серая пунктирная) - текущая практика большинства банков.
- **Цветные кривые** (uplift-модели) - то, что мы предлагаем.

Пространство между risk-based и oracle - это **потенциальный выигрыш** от перехода
на uplift-подход. Цель каждого meta-learner - заполнить этот gap.

**Нисходящий участок** у некоторых кривых в правой части - эффект 'sleeping dogs':
при расширении выбора мы начинаем включать клиентов, которым коммуникация вредит.


In [ ]:
def compute_budget_curve(df, score, n_points=200):
    # Вычисляет кривую: [доля_отобранных, накопленное_снижение_PD, накопленная_стоимость]
    df_sorted = df.assign(score=score).sort_values('score', ascending=False).reset_index(drop=True)
    n = len(df_sorted)
    step = max(1, n // n_points)

    fracs, prevented, costs = [0.0], [0.0], [0.0]
    for k in range(step, n + 1, step):
        top_k = df_sorted.iloc[:k]
        total_prevented = top_k['prevented_default_prob'].sum()
        total_cost = k * COST_ANY_CONTACT   # усреднённая стоимость контакта
        fracs.append(k / n)
        prevented.append(total_prevented)
        costs.append(total_cost)
    return np.array(fracs), np.array(prevented), np.array(costs)


# Вычисляем кривые для всех стратегий
budget_curves = {}
for name, score in strategies.items():
    fracs, prevented, costs = compute_budget_curve(df_test, score)
    budget_curves[name] = (fracs, prevented, costs)

print('Кривые вычислены для всех стратегий.')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Левый график: предотвращённые дефолты vs % охвата ────────────────────────
ax = axes[0]
for name, (fracs, prevented, costs) in budget_curves.items():
    lw = 2.5 if name in ('Oracle', 'Risk-based (BASE_PD)') else 1.8
    ax.plot(fracs * 100, prevented,
            label=name, color=strategy_colors[name],
            linestyle=strategy_styles[name], linewidth=lw)

ax.set_xlabel('Доля клиентов, получивших коммуникацию (%)')
ax.set_ylabel('Суммарное снижение вероятности дефолтов')
ax.set_title('Предотвращённые дефолты vs охват клиентов')
ax.legend(loc='upper left', fontsize=8)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

# ── Правый график: предотвращённые дефолты vs бюджет ─────────────────────────
ax = axes[1]
n_test = len(df_test)
for name, (fracs, prevented, costs) in budget_curves.items():
    lw = 2.5 if name in ('Oracle', 'Risk-based (BASE_PD)') else 1.8
    budget_mln = costs / 1_000_000  # в миллионах рублей
    ax.plot(budget_mln, prevented,
            label=name, color=strategy_colors[name],
            linestyle=strategy_styles[name], linewidth=lw)

ax.set_xlabel('Бюджет на коммуникации (млн руб.)')
ax.set_ylabel('Суммарное снижение вероятности дефолтов')
ax.set_title('Предотвращённые дефолты vs бюджет')
ax.legend(loc='upper left', fontsize=8)

plt.suptitle('Эффективность стратегий коммуникации (test выборка)', fontsize=13)
plt.tight_layout()
plt.show()

print()
print('Как читать:')
print('  - Чем выше кривая - тем больше дефолтов предотвращает стратегия при данном охвате.')
print('  - Oracle - максимально возможный результат.')
print('  - Gap между Risk-based и Oracle - потенциальный выигрыш от uplift-подхода.')


## 5. Сравнение стратегий при фиксированных уровнях охвата

На практике банк не выбирает 'всех или никого' - он задаёт **CRM-бюджет**.
Рассмотрим три реалистичных сценария: контакт с 10%, 20% и 30% клиентов.

Для каждого сценария считаем:
1. **Снижение числа ожидаемых дефолтов** (в штуках и процентах от baseline)
2. **Экономический эффект** (снижение * LGD)
3. **ROI** (эффект / затраты)

Это переводит абстрактный AUUC в **рублёвый результат**, понятный бизнесу.


In [ ]:
def policy_at_k(df, score, k_frac, cost_per_contact=COST_ANY_CONTACT):
    # Возвращает метрики стратегии при охвате k_frac (доля от всех клиентов)
    k = int(len(df) * k_frac)
    if k == 0:
        return {'n_contacted': 0, 'prevented_pd': 0, 'total_cost': 0, 'roi': 0}

    top_k = df.assign(score=score).sort_values('score', ascending=False).iloc[:k]

    prevented_pd = top_k['prevented_default_prob'].sum()
    economic_value = prevented_pd * LOSS_GIVEN_DEFAULT
    total_cost = k * cost_per_contact
    roi = (economic_value - total_cost) / total_cost if total_cost > 0 else 0

    return {
        'n_contacted':    k,
        'prevented_pd':   prevented_pd,
        'econ_value_mln': economic_value / 1_000_000,
        'total_cost_mln': total_cost / 1_000_000,
        'roi':            roi,
    }


# Считаем метрики для трёх уровней охвата
for k_frac in [0.10, 0.20, 0.30]:
    print(f'\n=== Охват {k_frac:.0%} клиентов ===')
    rows = []
    for name, score in strategies.items():
        if name == 'Никого не контактировать':
            continue
        m = policy_at_k(df_test, score, k_frac)
        rows.append({
            'Стратегия':         name,
            'Снижение PD':       m['prevented_pd'],
            'Экон. эффект, млн': m['econ_value_mln'],
            'Затраты, млн':      m['total_cost_mln'],
            'ROI':               m['roi'],
        })

    df_k = pd.DataFrame(rows).set_index('Стратегия')
    # Добавляем % от oracle
    oracle_val = df_k.loc['Oracle', 'Снижение PD']
    df_k['% от Oracle'] = (df_k['Снижение PD'] / oracle_val * 100).round(1)
    print(df_k.round(4).to_string())


## 6. ROI-кривая: при каком охвате стратегия окупается?

**ROI** (Return on Investment) показывает: на каждый вложенный рубль сколько рублей
экономии на дефолтах мы получаем?

$$\text{ROI} = \frac{\text{Снижение дефолтов} \cdot \text{LGD} - \text{Затраты}}{\text{Затраты}}$$

ROI > 0 означает, что коммуникационная кампания прибыльна.

**Ключевой вопрос:** при каком охвате каждая стратегия переходит из 'убыточной' в 'прибыльную'?
Uplift-модель должна иметь более низкий 'порог безубыточности' и более высокий пиковый ROI.


In [ ]:
def compute_roi_curve(df, score, n_points=200, cost_per_contact=COST_ANY_CONTACT):
    # Кривая ROI при увеличении охвата.
    df_sorted = df.assign(score=score).sort_values('score', ascending=False).reset_index(drop=True)
    n = len(df_sorted)
    step = max(1, n // n_points)

    fracs, roi_vals = [0.0], [0.0]
    for k in range(step, n + 1, step):
        top_k = df_sorted.iloc[:k]
        prevented = top_k['prevented_default_prob'].sum()
        econ_value = prevented * LOSS_GIVEN_DEFAULT
        cost = k * cost_per_contact
        roi = (econ_value - cost) / cost if cost > 0 else 0
        fracs.append(k / n)
        roi_vals.append(roi)
    return np.array(fracs), np.array(roi_vals)


fig, ax = plt.subplots(figsize=(12, 6))

for name, score in strategies.items():
    if name == 'Никого не контактировать':
        continue
    fracs, roi_vals = compute_roi_curve(df_test, score)
    lw = 2.5 if name in ('Oracle', 'Risk-based (BASE_PD)') else 1.8
    ax.plot(fracs * 100, roi_vals,
            label=name, color=strategy_colors[name],
            linestyle=strategy_styles[name], linewidth=lw)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_xlabel('Доля клиентов, получивших коммуникацию (%)')
ax.set_ylabel('ROI (доход / затраты - 1)')
ax.set_title('ROI коммуникационной кампании vs охват клиентов')
ax.legend(loc='upper right', fontsize=8)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}x'))
plt.tight_layout()
plt.show()

print()
print('Где кривая пересекает 0 - точка безубыточности кампании.')
print('Чем выше пик кривой - тем выгоднее стратегия при оптимальном бюджете.')


## 7. Multi-treatment policy: какой канал выбрать для каждого клиента?

До сих пор мы сравнивали стратегии в формате 'контактировать / не контактировать' (бинарный treatment).
Но в реальности банк **выбирает канал**: SMS, автозвонок или оператор.

Каждый канал имеет разную стоимость и разный эффект. Оптимальная политика:

$$\pi^*(x) = \arg\max_{t \in \{\text{control, sms, robot, operator}\}} \left[ -\hat{\tau}_t(x) \cdot V_{\text{default}} - C_t \right]$$

Иными словами: выбираем канал с наибольшим **ожидаемым чистым эффектом**.
Если для клиента выгоднее 'не контактировать' - не контактируем.

### Что мы ожидаем увидеть

- Дешёвые каналы (SMS) - для клиентов с умеренным предсказанным эффектом.
- Дорогие каналы (оператор) - только для клиентов с высоким предсказанным эффектом,
  где дорогая коммуникация окупается.
- 'Не контактировать' - для sleeping dogs и sure things.


In [ ]:
# Multi-treatment policy на основе T-Learner (per-channel uplift)
# T-Learner у нас даёт uplift по каждому каналу отдельно

# Ожидаемый чистый эффект = (-uplift) * LGD - cost
# Нормализуем LGD к числу клиентов для удобства: используем relative units

def compute_optimal_channel(df, uplift_cols, costs_dict, lgd=LOSS_GIVEN_DEFAULT):
    # Вычисляет оптимальный канал для каждого клиента.
    # uplift_cols: словарь {channel_name: column_name_in_df}
    # costs_dict:  словарь {channel_name: cost}

    net_values = {'control': pd.Series(np.zeros(len(df)), index=df.index)}

    for ch, col in uplift_cols.items():
        # net_value = -uplift * LGD - cost
        # (отрицательный uplift = снижение PD = хорошо, поэтому минус даёт положительное)
        net_values[ch] = -df[col] * lgd - costs_dict[ch]

    net_df = pd.DataFrame(net_values)
    # Оптимальный канал - с наибольшим net_value
    optimal_channel = net_df.idxmax(axis=1)
    optimal_net_value = net_df.max(axis=1)

    return optimal_channel, optimal_net_value, net_df


# Per-channel uplift S-Learner
uplift_cols_s = {
    'sms':          'uplift_s_sms',
    'robot_call':   'uplift_s_robot',
    'operator_call':'uplift_s_operator',
}
costs_per_client = {
    'sms':          COMM_COSTS['sms'],
    'robot_call':   COMM_COSTS['robot_call'],
    'operator_call':COMM_COSTS['operator_call'],
}

opt_channel_s, opt_value_s, net_df_s = compute_optimal_channel(
    df_test, uplift_cols_s, costs_per_client
)

print('Оптимальное распределение каналов (S-Learner policy):')
ch_dist = opt_channel_s.value_counts(normalize=True).mul(100).round(1)
print(ch_dist.astype(str) + '%')

print(f'\nОжидаемый суммарный чистый эффект: {opt_value_s.sum()/1_000_000:.2f} млн руб.')
print(f'Ожидаемый чистый эффект на клиента: {opt_value_s.mean():.0f} руб.')


In [ ]:
# Аналогично для T-Learner
uplift_cols_t = {
    'sms':          'uplift_t_sms',
    'robot_call':   'uplift_t_robot',
    'operator_call':'uplift_t_operator',
}

opt_channel_t, opt_value_t, net_df_t = compute_optimal_channel(
    df_test, uplift_cols_t, costs_per_client
)

print('Оптимальное распределение каналов (T-Learner policy):')
print(opt_channel_t.value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

# Сравниваем с истинным оптимальным распределением (oracle)
print('\nИстинное оптимальное распределение каналов (Oracle, ORACLE_COMMUNICATION):')
if 'ORACLE_COMMUNICATION' in pd.read_csv('data/processed/uplift-dataset.csv').columns:
    # Загрузим из полного датасета
    df_full = pd.read_csv('data/processed/uplift-dataset.csv')
    n_test = len(df_test)
    oot_size = int(len(df_full) * 0.2)
    test_start = oot_size + int((len(df_full) - oot_size) * 0.75)
    oracle_comm = df_full.iloc[test_start:test_start + n_test]['ORACLE_COMMUNICATION'].reset_index(drop=True)
    print(oracle_comm.value_counts(normalize=True).mul(100).round(1).astype(str) + '%')
else:
    print('  (ORACLE_COMMUNICATION недоступна в uplift_scores.csv)')


In [ ]:
# Визуализация: распределение каналов по сегментам риска
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, opt_ch) in zip(axes, [('S-Learner', opt_channel_s), ('T-Learner', opt_channel_t)]):
    ch_by_seg = pd.crosstab(df_test['RISK_SEGMENT'], opt_ch, normalize='index') * 100
    # Порядок сегментов
    seg_order = ['low_risk', 'medium_risk', 'high_risk']
    ch_by_seg = ch_by_seg.reindex(seg_order, fill_value=0)

    ch_by_seg.plot(kind='bar', ax=ax, width=0.7, colormap='Set2')
    ax.set_title(f'Оптимальный канал по сегменту риска\n({name} policy)')
    ax.set_xlabel('Сегмент риска')
    ax.set_ylabel('Доля клиентов (%)')
    ax.legend(title='Канал', fontsize=8)
    ax.set_xticklabels(['Низкий риск', 'Средний риск', 'Высокий риск'], rotation=0)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

plt.suptitle('Какой канал выбирает оптимальная политика для каждого сегмента?', fontsize=12)
plt.tight_layout()
plt.show()

print('Интерпретация: соответствует ли распределение каналов интуиции?')
print('  high_risk -> operator_call (дорого, но эффективно при высоком BASE_PD)')
print('  low_risk  -> sms (дёшево, при маленьком BASE_PD operator не окупается)')


## 8. Сегментация клиентов: Persuadables, Sure Things, Sleeping Dogs, Lost Causes

Классическая сегментация в uplift-моделировании делит клиентов на 4 квадранта:

| | **Без коммуникации: платит** | **Без коммуникации: дефолт** |
|---|---|---|
| **С коммуникацией: платит** | Sure Thing (не нужна коммуникация) | **Persuadable** (цель!) |
| **С коммуникацией: дефолт** | **Sleeping Dog** (избегать!) | Lost Cause (бесполезно) |

**Цель uplift-модели:** находить Persuadables и избегать Sleeping Dogs.

В синтетических данных мы можем вычислить истинный квадрант через `TRUE_UPLIFT` и `PD_NO_CONTACT`.

- **Persuadable**: TRUE_UPLIFT < -порог (коммуникация значимо снижает PD)
- **Sleeping Dog**: TRUE_UPLIFT > +порог (коммуникация значимо повышает PD)
- **Sure Thing**: TRUE_UPLIFT ≈ 0, BASE_PD < медиана
- **Lost Cause**: TRUE_UPLIFT ≈ 0, BASE_PD > медиана


In [ ]:
# Пороговое значение для классификации эффекта как 'значимого'
UPLIFT_THRESHOLD = df_test['TRUE_UPLIFT'].std() * 0.5
print(f'Порог для Persuadable/Sleeping Dog: {UPLIFT_THRESHOLD:.5f}')

# Сегментация по TRUE_UPLIFT
df_test_seg = df_test.copy()

conditions = [
    df_test_seg['TRUE_UPLIFT'] < -UPLIFT_THRESHOLD,                         # Persuadable
    df_test_seg['TRUE_UPLIFT'] >  UPLIFT_THRESHOLD,                         # Sleeping Dog
    (df_test_seg['TRUE_UPLIFT'].abs() <= UPLIFT_THRESHOLD) &
    (df_test_seg['BASE_PD'] < df_test_seg['BASE_PD'].median()),              # Sure Thing
]
labels = ['Persuadable', 'Sleeping Dog', 'Sure Thing']
df_test_seg['segment'] = np.select(conditions, labels, default='Lost Cause')

print('\nРаспределение клиентов по сегментам (test):')
print(df_test_seg['segment'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

print('\nСредний TRUE_UPLIFT по сегментам:')
print(df_test_seg.groupby('segment')['TRUE_UPLIFT'].mean().round(5))


In [ ]:
# Насколько хорошо каждая модель идентифицирует Persuadables?
# Оцениваем: среди top-K% по скору - какая доля реальных Persuadables?

persuadable_mask = (df_test_seg['segment'] == 'Persuadable').values
sleeping_dog_mask = (df_test_seg['segment'] == 'Sleeping Dog').values

print('Качество идентификации Persuadables (top-20% по скору):')
print(f'{'Стратегия':<30} {'Persuadables, %':>18} {'Sleeping Dogs, %':>18}')
print('-' * 68)

k20 = int(len(df_test) * 0.20)
for name, score in strategies.items():
    if name == 'Никого не контактировать':
        continue
    top_k_idx = np.argsort(-score)[:k20]
    pct_persuadable  = persuadable_mask[top_k_idx].mean() * 100
    pct_sleeping_dog = sleeping_dog_mask[top_k_idx].mean() * 100
    print(f'{name:<30} {pct_persuadable:>17.1f}% {pct_sleeping_dog:>17.1f}%')

print(f'\nВ популяции: Persuadables={persuadable_mask.mean()*100:.1f}%, '
      f'Sleeping Dogs={sleeping_dog_mask.mean()*100:.1f}%')


In [ ]:
# Визуализация: pie chart сегментов + violin plot uplift по сегментам
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
seg_counts = df_test_seg['segment'].value_counts()
colors_seg = {'Persuadable': 'steelblue', 'Sure Thing': 'lightgreen',
              'Sleeping Dog': 'salmon', 'Lost Cause': 'lightgray'}
axes[0].pie(seg_counts.values, labels=seg_counts.index,
            colors=[colors_seg[s] for s in seg_counts.index],
            autopct='%1.1f%%', startangle=90)
axes[0].set_title('Структура клиентской базы\nпо реакции на коммуникацию')

# Violin plot TRUE_UPLIFT по сегментам
seg_order = ['Persuadable', 'Sure Thing', 'Lost Cause', 'Sleeping Dog']
df_violin = df_test_seg[df_test_seg['segment'].isin(seg_order)]
sns.violinplot(data=df_violin, x='segment', y='TRUE_UPLIFT',
               order=seg_order, palette=colors_seg,
               ax=axes[1], inner='box', cut=0)
axes[1].axhline(0, color='red', linewidth=0.8, linestyle='--')
axes[1].axhline(-UPLIFT_THRESHOLD, color='blue', linewidth=0.8, linestyle=':',
                label=f'Порог ({-UPLIFT_THRESHOLD:.4f})')
axes[1].axhline(UPLIFT_THRESHOLD, color='orange', linewidth=0.8, linestyle=':')
axes[1].set_title('Распределение TRUE_UPLIFT\nпо сегментам клиентов')
axes[1].set_xlabel('')
axes[1].set_ylabel('TRUE_UPLIFT')
axes[1].legend(fontsize=8)

plt.suptitle('Сегментация клиентов: Persuadable / Sleeping Dog / Sure Thing / Lost Cause',
             fontsize=11)
plt.tight_layout()
plt.show()


## 9. Bootstrap доверительные интервалы

Метрики, вычисленные на тестовой выборке, подвержены случайному шуму выборки.
Bootstrap позволяет оценить **статистическую значимость** различий между стратегиями.

**Методология:**
1. Перевыборка с возвращением: берём N случайных подвыборок из test.
2. Для каждой подвыборки вычисляем эффект стратегии при фиксированном охвате (20%).
3. Строим 95% доверительный интервал (2.5-й и 97.5-й персентили).

Если CI двух стратегий не пересекаются - разница статистически значима.


In [ ]:
N_BOOTSTRAP = 500
K_FRAC      = 0.20   # фиксированный охват для bootstrap

rng_boot = np.random.RandomState(RANDOM_SEED)

bootstrap_results = {name: [] for name in strategies if name != 'Никого не контактировать'}

print(f'Bootstrap ({N_BOOTSTRAP} итераций, охват={K_FRAC:.0%})...')
for b in range(N_BOOTSTRAP):
    idx = rng_boot.choice(len(df_test), size=len(df_test), replace=True)
    df_b = df_test.iloc[idx].reset_index(drop=True)

    for name, score in strategies.items():
        if name == 'Никого не контактировать':
            continue
        score_b = score[idx]
        m = policy_at_k(df_b, score_b, K_FRAC)
        bootstrap_results[name].append(m['prevented_pd'])

    if (b + 1) % 100 == 0:
        print(f'  {b+1}/{N_BOOTSTRAP}', end='\r')

print('\nBootstrap завершён.')
print(f'\n95% CI снижения PD при охвате {K_FRAC:.0%}:')
print(f'{'Стратегия':<30} {'Среднее':>10} {'CI 2.5%':>10} {'CI 97.5%':>10}')
print('-' * 65)
boot_summary = {}
for name, vals in bootstrap_results.items():
    arr = np.array(vals)
    boot_summary[name] = (arr.mean(), np.percentile(arr, 2.5), np.percentile(arr, 97.5))
    m, lo, hi = boot_summary[name]
    print(f'{name:<30} {m:>10.4f} {lo:>10.4f} {hi:>10.4f}')


In [ ]:
# Визуализация Bootstrap CI
fig, ax = plt.subplots(figsize=(10, 5))

names = list(boot_summary.keys())
means = [boot_summary[n][0] for n in names]
lows  = [boot_summary[n][1] for n in names]
highs = [boot_summary[n][2] for n in names]
errors_lo = [m - lo for m, lo in zip(means, lows)]
errors_hi = [hi - m for m, hi in zip(means, highs)]

y_pos = range(len(names))
colors = [strategy_colors[n] for n in names]

ax.barh(y_pos, means, xerr=[errors_lo, errors_hi],
        color=colors, alpha=0.8, capsize=5, error_kw={'elinewidth': 1.5})
ax.set_yticks(y_pos)
ax.set_yticklabels(names)
ax.set_xlabel('Снижение PD (суммарное, top-20% клиентов)')
ax.set_title(f'Bootstrap 95% CI сравнения стратегий (N={N_BOOTSTRAP})')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print()
print('Если CI двух стратегий не пересекаются - разница статистически значима (p < 0.05).')


## 10. Итоговые выводы и практические рекомендации

### Что мы узнали

**Основной результат** этого ноутбука - перевод абстрактных AUUC-метрик в конкретные бизнес-цифры.

#### Сравнение стратегий

| Стратегия | Сильные стороны | Слабые стороны |
|---|---|---|
| Risk-based | Прост, не требует специальных данных | Контактирует 'sure things', не видит 'sleeping dogs' |
| S-Learner | Один pass обучения | Может потерять treatment effect при слабом сигнале |
| T-Learner | Специализация на канале | Чувствителен к дисбалансу групп |
| X-Learner | Устойчив при дисбалансе | Требует двух этапов обучения |
| DR-Learner | Двойная робастность | Требует кросс-валидации, самый дорогой в обучении |

#### Когда стоит применять uplift-подход?

По результатам наших экспериментов:

1. **При ограниченном бюджете** (охват < 30%) - uplift-модель выигрывает у risk-based,
   так как точнее находит Persuadables.

2. **При значительном selection bias** - DR-Learner предпочтительнее S/T-Learner.

3. **При multi-treatment задаче** (несколько каналов с разной стоимостью) -
   per-channel uplift необходим для оптимизации бюджета.

4. **При слабом uplift-сигнале (SNR < 1)** - различия между подходами статистически
   незначимы; risk-based может оказаться достаточным.

#### Ограничения исследования

1. Синтетический датасет: TRUE_UPLIFT сгенерирован по заданным правилам,
   реальный CRM-эффект может иметь иную структуру.
2. LGD и стоимости коммуникаций - оценочные параметры; чувствительность к ним
   требует отдельного анализа (sensitivity analysis).
3. Bootstrap CI оценивают дисперсию выборки, но не учитывают ошибки самой генерации данных.

### Дальнейшие шаги

- **sensitivity_analysis.ipynb**: как меняются выводы при разных параметрах SNR и selection bias?
- **Внешний датасет**: проверка методологии на реальных uplift-данных (Hillstrom, Lenta).
